In [ ]:
import os
import subprocess

import pandas as pd
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18
from torch.utils.data import DataLoader, Dataset

import random
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
if DEVICE != 'cuda':
    raise RuntimeError('Make sure you have added an accelerator to your notebook; the submission will fail otherwise!')

In [ ]:
def load_example(df_row):
    image = torchvision.io.read_image(df_row['image_path'])
    result = {
        'image': image,
        'image_id': df_row['image_id'],
        'age_group': df_row['age_group'],
        'age': df_row['age'],
        'person_id': df_row['person_id']
    }
    return result


class HiddenDataset(Dataset):
    '''The hidden dataset.'''
    def __init__(self, split='train'):
        super().__init__()
        self.examples = []

        df = pd.read_csv(f'/kaggle/input/neurips-2023-machine-unlearning/{split}.csv')
        df['image_path'] = df['image_id'].apply(
            lambda x: os.path.join('/kaggle/input/neurips-2023-machine-unlearning/', 'images', x.split('-')[0], x.split('-')[1] + '.png'))
        df = df.sort_values(by='image_path')
        df.apply(lambda row: self.examples.append(load_example(row)), axis=1)
        if len(self.examples) == 0:
            raise ValueError('No examples.')

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]
        image = example['image']
        image = image.to(torch.float32)
        example['image'] = image
        return example


def get_dataset(batch_size):
    '''Get the dataset.'''
    retain_ds = HiddenDataset(split='retain')
    forget_ds = HiddenDataset(split='forget')
    val_ds = HiddenDataset(split='validation')

    retain_loader = DataLoader(retain_ds, batch_size=batch_size, shuffle=True)
    forget_loader = DataLoader(forget_ds, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=True)

    return retain_loader, forget_loader, validation_loader

def get_val_loader(batch_size):
    val_ds = HiddenDataset(split='validation')
    samples = []
    for sample in val_ds:
        y = sample["age_group"]
        if int(y) == 0 or int(y) == 1:
            samples.append(sample)
    val_loader = DataLoader(samples, batch_size=batch_size, shuffle=True)
    return val_loader

In [ ]:
def kl_loss(model_logits, teacher_logits, temperature=1.):
    '''
    Calculate the Kullback-Leibler (KL) divergence loss between the output of a model (student) and the output of a teacher model.
    
    Args:
        model_logits (torch.Tensor): The logits from the student model. Logits are the raw outputs of the last neural network layer prior to softmax activation.
        teacher_logits (torch.Tensor): The logits from the teacher model.
        temperature (float, optional): A temperature parameter that scales the logits before applying softmax. Higher temperatures produce softer probability distributions. Defaults to 1.

    Returns:
        torch.Tensor: The KL divergence loss, averaged over the batch.
    '''
    # apply softmax to the (scaled) logits of the teacher model
    teacher_output_softmax = F.softmax(teacher_logits / temperature, dim=1) 
    # apply log softmax to the (scaled) logits of the student model
    output_log_softmax = F.log_softmax(model_logits / temperature, dim=1)
    
    # calculate the KL divergence between the student and teacher outputs
    kl_div = F.kl_div(output_log_softmax, teacher_output_softmax, reduction='batchmean')
    return kl_div

In [ ]:
def soft_cross_entropy(preds, soft_targets):
    '''
    Calculate the soft cross-entropy loss between predictions and soft targets.

    Args:
        preds (torch.Tensor): The predictions from the student model.
        soft_targets (torch.Tensor): The soft targets, the probability distribution from a teacher model.

    Returns:
        torch.Tensor: The average soft cross-entropy loss across all instances in the batch.
    '''
    # calculate the element-wise cross-entropy loss
    loss = torch.sum(-soft_targets * torch.log_softmax(preds, dim=1), dim=1)
    return torch.mean(loss)

In [ ]:
def unlearning(model, teacher_model, retain_loader, forget_loader, val_loader, temperature=1.,
               weight=None, device='cuda'):
    '''
    Perform unlearning on a model by resetting specific layers and then retraining
    with a combination of knowledge distillation and fine-tuning.

    The process involves:
    1) resetting certain layers of the model, 
    2) warming up the model using knowledge distillation from a teacher model using the validation set and
    3) fine-tuning the model on the retain set.

    Args:
        model (torch.nn.Module): The student model to be unlearned and retrained.
        teacher_model (torch.nn.Module): The teacher model providing guidance in the knowledge distillation process.
        retain_loader (DataLoader): DataLoader for the dataset to retain knowledge about.
        forget_loader (DataLoader): DataLoader for the dataset to forget (not used in this function).
        val_loader (DataLoader): DataLoader for the validation set used during the warm-up phase.
        temperature (float, optional): Temperature parameter in knowledge distillation. Defaults to 1.
        weight (torch.Tensor, optional): A manual rescaling weight given to each class for cross-entropy loss. Defaults to None.
        device (str, optional): Device to run the model on ('cuda' or 'cpu'). Defaults to 'cuda'.

    Returns:
        None: This function modifies the model in-place.
    '''
    # reset the first convolutional layer and the final fully connected layer of the model
    model.conv1.reset_parameters()
    model.fc.reset_parameters()
    
    # set the teacher model to evaluation mode
    teacher_model.eval()
    
    # define the number of epochs for warm-up and retain phases
    retain_epochs = 3
    warmup_epochs = 3
    
    # standard cross-entropy loss for fine-tuning
    criterion = nn.CrossEntropyLoss(weight=weight)
    
    # Warm-up phase: Adjust the student model closer to the teacher model using knowledge distillation
    optimizer = torch.optim.SGD(model.parameters(), lr=9e-4, weight_decay=5e-4, momentum=0.9)
    for epoch in range(warmup_epochs):

        model.train()
        for sample in val_loader:
            x = sample["image"]
            y = sample["age_group"]
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            student_out = model(x)
            
            with torch.no_grad():
                teacher_out = teacher_model(x)
                
            loss = kl_loss(model_logits=student_out, 
                           teacher_logits=teacher_out, 
                           temperature=temperature)
            loss.backward()
            optimizer.step()

    # Fine-tuning phase: Train the model on the retain set using standard cross-entropy along with knowledge distillation
    optimizer = torch.optim.SGD(model.parameters(),
                                lr=1e-3, weight_decay=5e-4, momentum=0.9)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=retain_epochs
    )
    for epoch in range(retain_epochs):
        model.train()
        for sample in retain_loader:
            x = sample["image"]
            y = sample["age_group"]
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            with torch.no_grad():
                teacher_out = teacher_model(x)
            soft_labels = torch.softmax(teacher_out / temperature, dim=1)
            soft_predictions = torch.log_softmax(out / temperature, dim=1)
            loss = soft_cross_entropy(soft_predictions, soft_labels)
            loss += criterion(out, y)
            loss += kl_loss(model_logits=out, 
                           teacher_logits=teacher_out, 
                           temperature=temperature)
            loss.backward()
            optimizer.step()
        scheduler.step()

    model.eval()

In [ ]:
if os.path.exists('/kaggle/input/neurips-2023-machine-unlearning/empty.txt'):
    # mock submission
    subprocess.run('touch submission.zip', shell=True)
else:
    # Note: it's really important to create the unlearned checkpoints outside of the working directory 
    # as otherwise this notebook may fail due to running out of disk space.
    # The below code saves them in /kaggle/tmp to avoid that issue.
    os.makedirs('/kaggle/tmp', exist_ok=True)
    retain_loader, forget_loader, _ = get_dataset(64)
    
    validation_loader = get_val_loader(64) # Validation samples only from classes 0,1 
    
    teacher_model = resnet18(weights=None, num_classes=10) # instantiate teacher 
    teacher_model.load_state_dict(torch.load('/kaggle/input/neurips-2023-machine-unlearning/original_model.pth')) # Initialize teacher with the original model
    teacher_model.to(DEVICE)
    
    model = resnet18(weights=None, num_classes=10)
    model.to(DEVICE)
    for i in range(512):
        model.load_state_dict(torch.load('/kaggle/input/neurips-2023-machine-unlearning/original_model.pth')) # student model
        unlearning(model, teacher_model, retain_loader, forget_loader, validation_loader, temperature=5.,
                   weight=None, device=DEVICE) # perform unlearning
        state = model.state_dict()
        torch.save(state, f'/kaggle/tmp/unlearned_checkpoint_{i}.pth')
        
    # Ensure that submission.zip will contain exactly 512 checkpoints 
    # (if this is not the case, an exception will be thrown).
    unlearned_ckpts = os.listdir('/kaggle/tmp')
    if len(unlearned_ckpts) != 512:
        raise RuntimeError('Expected exactly 512 checkpoints. The submission will throw an exception otherwise.')
        
    subprocess.run('zip submission.zip /kaggle/tmp/*.pth', shell=True)